# Notebook 03 — RAG evaluation (small benchmark)

Goal: stop judging RAG by vibes.

We’ll:
- load a small eval set (`data/eval/eval_set.jsonl`)
- run retrieval for each question
- compute **simple measurable signals**:
  - whether retrieved context contains expected key phrases (proxy for retrieval recall)
  - whether the final answer contains expected key phrases (proxy for usefulness)

This is not a perfect evaluation, but it builds the engineer habit: **measure, then tune**.


In [ ]:
!pip -q install langchain langchain-community langchain-text-splitters faiss-cpu numpy sentence-transformers

In [ ]:
import json
import re
from pathlib import Path
import sys

PROJECT_ROOT = Path('/content/learn RAG')
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / 'src'))

from rag_core import load_text_docs, build_chunks

docs = load_text_docs(PROJECT_ROOT / 'data' / 'docs')
eval_path = PROJECT_ROOT / 'data' / 'eval' / 'eval_set.jsonl'
eval_items = [json.loads(line) for line in eval_path.read_text(encoding='utf-8').splitlines() if line.strip()]
print('docs:', len(docs))
print('eval items:', len(eval_items))


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

def make_retriever(chunk_chars: int, overlap_chars: int, k: int = 5):
    chunks = build_chunks(docs, chunk_chars=chunk_chars, overlap_chars=overlap_chars)
    vs = FAISS.from_texts(
        texts=[c.text for c in chunks],
        embedding=embeddings,
        metadatas=[c.metadata for c in chunks],
    )
    return vs.as_retriever(search_kwargs={'k': k})

def grounded_local_answer(question: str, retrieved_docs):
    context = "\n\n".join([d.page_content for d in retrieved_docs]).strip()
    if not context:
        return "I don't know based on the available documents."
    q_tokens = set(re.findall(r"[a-zA-Z]{3,}", question.lower()))
    sents = re.split(r"(?<=[.!?])\s+", context)
    scored = []
    for s in sents:
        st = set(re.findall(r"[a-zA-Z]{3,}", s.lower()))
        scored.append((len(q_tokens & st), s))
    scored.sort(key=lambda x: x[0], reverse=True)
    best = [s for score, s in scored[:4] if score > 0]
    if not best:
        best = [s for _, s in scored[:2] if s.strip()]
    return " ".join([b.strip() for b in best]).strip()


## Benchmark runner

We’ll compare two chunk configs and see which gets better retrieval/answer coverage.


In [ ]:
def contains_all(haystack: str, needles):
    h = haystack.lower()
    return all(n.lower() in h for n in needles)

def run_benchmark(chunk_chars: int, overlap_chars: int, k: int = 5):
    retriever = make_retriever(chunk_chars=chunk_chars, overlap_chars=overlap_chars, k=k)
    rows = []
    for item in eval_items:
        q = item['question']
        expected = item.get('expected_answer_contains', [])

        hits = retriever.get_relevant_documents(q)
        context = "\n\n".join([h.page_content for h in hits])
        answer = grounded_local_answer(q, hits)

        retrieval_ok = contains_all(context, expected) if expected else True
        answer_ok = contains_all(answer, expected) if expected else True

        rows.append({
            'id': item.get('id'),
            'question': q,
            'retrieval_ok': retrieval_ok,
            'answer_ok': answer_ok,
            'expected': expected,
        })

    retrieval_acc = sum(r['retrieval_ok'] for r in rows) / max(1, len(rows))
    answer_acc = sum(r['answer_ok'] for r in rows) / max(1, len(rows))
    return rows, retrieval_acc, answer_acc

configs = [
    {'chunk_chars': 900, 'overlap_chars': 120, 'k': 5},
    {'chunk_chars': 1200, 'overlap_chars': 150, 'k': 5},
]

for cfg in configs:
    rows, retr_acc, ans_acc = run_benchmark(**cfg)
    print('\nCONFIG:', cfg)
    print('retrieval_ok rate:', round(retr_acc, 3))
    print('answer_ok rate:', round(ans_acc, 3))
    for r in rows:
        if not r['retrieval_ok'] or not r['answer_ok']:
            print('  FAIL:', r['id'], '| retrieval_ok=', r['retrieval_ok'], '| answer_ok=', r['answer_ok'])


## What to do next (real evaluation skill)
- Replace the sample eval with **20–30 Q/A pairs from your real docs**.
- Add a column for human labels: correct/incorrect + notes.
- Track changes when you adjust chunking, k, or metadata.
- If your environment allows, add an LLM judge (RAGAS/DeepEval) later.
